In [1]:
import json
import re



In [2]:
class ReflectionAgent:
    def __init__(self, tree_data):
        self.tree = {node['id']: node for node in tree_data}
        self.state = {
            "answers": {},
            "signals": {"axis1": {"internal": 0, "external": 0}, 
                        "axis2": {"contribution": 0, "entitlement": 0}, 
                        "axis3": {"self": 0, "altro": 0}}
        }

    def interpolate_text(self, text):
        """Replaces {NodeID.answer} with the actual stored answer."""
        if not text: return ""
        matches = re.findall(r"\{(.*?)\.answer\}", text)
        for node_id in matches:
            val = self.state["answers"].get(node_id, "[unknown]")
            text = text.replace(f"{{{node_id}.answer}}", val)
        
        # Handle Summary logic for dominant signals
        for axis in ["axis1", "axis2", "axis3"]:
            counts = self.state["signals"][axis]
            dominant = max(counts, key=counts.get)
            text = text.replace(f"{{{axis}.dominant}}", dominant)
        return text

    def process_signals(self, signal_str):
        if not signal_str: return
        # Example: "axis1:internal"
        axis, pole = signal_str.split(":")
        self.state["signals"][axis][pole] += 1

    def run(self, start_node_id="START"):
        current_id = start_node_id
        
        while current_id:
            node = self.tree.get(current_id)
            if not node: break
            
            # 1. Process Signals
            self.process_signals(node.get("signal"))
            
            # 2. Handle Node Types
            node_type = node['type']
            text = self.interpolate_text(node.get('text', ''))

            if node_type in ["start", "reflection", "bridge", "summary"]:
                if text: print(f"\n{text}")
                if node_type == "reflection": input("[Press Enter to continue...]")
                current_id = node.get("target") or self.get_first_child(current_id)

            elif node_type == "question":
                print(f"\n{text}")
                options = node['options'].split("|")
                for i, opt in enumerate(options, 1):
                    print(f"{i}. {opt}")
                
                choice = int(input("\nPick an option (number): ")) - 1
                selected_text = options[choice]
                self.state["answers"][current_id] = selected_text
                current_id = self.get_first_child(current_id)

            elif node_type == "decision":
                # Logic: answer=ChoiceA|ChoiceB:TargetID
                rules = node['options'].split(";")
                last_answer = list(self.state["answers"].values())[-1]
                
                for rule in rules:
                    condition, target = rule.split(":")
                    valid_answers = condition.replace("answer=", "").split("|")
                    if last_answer in valid_answers:
                        current_id = target
                        break
            
            elif node_type == "end":
                print(f"\n{text}")
                break

    def get_first_child(self, parent_id):
        # Finds the first node in the list that lists this node as parent
        for node_id, node in self.tree.items():
            if node.get("parentId") == parent_id:
                return node_id
        return None



In [3]:
# --- EXECUTION ---
if __name__ == "__main__":
    # Load your Part A JSON file
    with open('reflection-decision-tree.json', 'r') as f:
        data = json.load(f)
    
    agent = ReflectionAgent(data)
    agent.run()

JSONDecodeError: Expecting value: line 4 column 3 (char 181)